# धडा 12 - एजंट स्क्रॅचपॅडसह चॅट इतिहास कमी करणे

हा नोटबुक Microsoft Agent Framework वापरून लांबणाऱ्या संभाषणांमध्ये संदर्भ कसा व्यवस्थापित करायचा हे दर्शवितो. संभाषणे वाढल्यावर टोकनांची संख्या वाढते — जी शेवटी मॉडेलच्या संदर्भ विंडोच्या मर्यादेपलीकडे जाऊ शकते. आम्ही यासाठी **संदर्भ संक्षेपण नमुना** आणि दीर्घकालीन स्मृतीसाठी **एजंट स्क्रॅचपॅड** वापरतो.

## आपण काय शिकणार आहात:
1. **संदर्भ व्यवस्थापन का महत्वाचे आहे**: टोकन मर्यादा आणि संदर्भ विंडो समजून घेणे
2. **संदर्भ-जागरूक एजंट**: स्वतःच्या संभाषण संदर्भाचे व्यवस्थापन करणारे एजंट तयार करणे
3. **संदर्भ संक्षेपण नमुना**: संभाषण इतिहास संक्षिप्त करण्यासाठी साधने वापरणे
4. **एजंट स्क्रॅचपॅड**: संदर्भ कमी केल्यानंतर राखली जाणारी स्मृती

## आवश्यक पूर्वतयारी:
- Azure OpenAI सेटअप ज्यात पर्यावरणीय चल (environment variables) कॉन्फिगर केलेले आहेत
- पूर्वीच्या धड्यांतून मूलभूत एजंट संकल्पना समजून घेणे


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## संदर्भ व्यवस्थापन का महत्त्वाचे आहे

प्रत्येक LLM कडे एक मर्यादित **संदर्भ विंडो** असते — एकाच विनंत्यामध्ये तो प्रक्रिया करू शकणाऱ्या टोकन्सची कमाल संख्या. एक बहु-टर्न संवाद पुढे जात असल्याने:

- **टोकन संख्या वर्धित होते** प्रत्येक वापरकर्त्याच्या संदेश आणि सहाय्यकाच्या प्रत्युत्तराबरोबर.
- **प्रॉम्प्ट टोकन्स खर्चावर प्रभुत्व ठेवतात** कारण प्रत्येक टर्नमध्ये संपूर्ण इतिहास पुन्हा पाठवला जातो.
- शेवटी संवाद **संदर्भ विंडो ओलांडतो** आणि मॉडेल कापणी करतो किंवा त्रुटी दर्शवतो.

### संदर्भ व्यवस्थापनासाठी धोरणे

| धोरण | कार्य करण्याची पद्धत | समायोजन |
|---|---|---|
| **कापणी** | सर्वात जुने संदेश काढून टाका | सुरुवातीचा संदर्भ हरवतो |
| **सारांश लेखन** | जुने संदेश संक्षेपात मांडणे | काही तपशील हरवतो, पण मुख्य मुद्दे जपले जातात |
| **स्क्रॅचपॅड / बाह्य स्मृती** | संभाषणाबाहेर मुख्य तथ्ये साठवणे | साधन कॉल आवश्यक, पण कोणत्याही कपातीला टिकून राहते |

या नोटबुकमध्ये आम्ही **सारांश लेखन** आणि **स्क्रॅचपॅड साधन** यांचा संयोजन करतो ज्यामुळे एजंट संवादाचा इतिहास संक्षेपित असतानाही सातत्य राखू शकतो.


## संदर्भ-जाणणारा एजंट तयार करणे


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## लांब संवादाचे अनुकरण करणे

चला एक बहु-टर्न संभाषण पाहूया ज्यातून कसे संदर्भ जमा होतात ते दिसेल. एजंटने महत्त्वाची माहिती (आव्हाने, बजेट, प्रवासाच्या तारखा) प्रत्येक टर्नदरम्यान ठेवावी आणि सलगता दाखवावी.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

एजंट कसे आधीच्या संवादातील संदर्भ ठेवतो ते लक्षात घ्या — त्याला जपान, सुशी, मंदिरे, छायाचित्रण, $3000 बजेट, एकटा प्रवास, आणि एप्रिलचा कालावधी याबद्दल माहित आहे. लहान संभाषणामध्ये हे चांगले कार्य करते, पण संवाद वाढल्यावर पूर्ण इतिहास पुन्हा पाठवणे महाग पडते.

संदर्भ संकलन पाहण्यासाठी आणखी संवाद घेऊया:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## संदर्भ संक्षेपणी नमुना

संवाद वाढत गेल्यामुळे, आपण एक **संक्षेपणी साधन** वापरू शकतो जे संचित संदर्भांना संक्षिप्त स्वरूपात रूपांतरित करते. एजंट या साधनाचा वापर करून मुख्य प्राधान्यांची नोंद करतो जेणेकरून जुने संदेश हटवले तरीही आवश्यक माहिती टिकून राहील.

हा नमुना अधिक परिष्कृत इतिहास कमी करण्यासाठीचा पाया आहे:
1. एजंट संवादातून मुख्य तथ्ये ओळखतो
2. ते टिकवण्यासाठी संक्षेपणी साधनाला कॉल करतो
3. जुने संदेश सुरक्षितपणे काढून टाकता येतात कारण संक्षेप जे महत्वाचे आहे ते घेतो

खाली आम्ही एक `summarize_preferences` साधन परिभाषित करतो ज्याला एजंट कॉल करून तो शिकलेल्या गोष्टींचा एक संक्षिप्त आढावा नोंदवू शकतो.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## सारांश

या धड्यात तुम्ही Microsoft Agent Framework वापरून दीर्घकाळ चालणाऱ्या एजंट संभाषणांमध्ये संदर्भ कसा व्यवस्थापित करायचा हे शिकले:

### मुख्य संकल्पना
- **संदर्भ विंडोज मर्यादित असतात** — संभाषण इतिहासातील प्रत्येक टोकनला खर्च येतो आणि तो मर्यादेकडे मोजला जातो.
- **सारांश साधने** एजंटला संचित संदर्भ संक्षिप्त सारांशांमध्ये रूपांतरित करण्याची परवानगी देतात, ज्यामुळे टोकन वापर कमी होतो आणि आवश्यक माहिती जपली जाते.
- **एजंट स्क्रॅचपॅड्स** दीर्घकाळ टिकणारे बाह्य स्मरणशक्ती प्रदान करतात जी कोणत्याही संभाषण कमी झाल्यावरही टिकून राहते.

### तुम्ही काय तयार केले
- एक **संदर्भ-जाणकार एजंट** जो बहु-टर्न संभाषणांमध्ये सलग राहतो
- एक **सारांश साधन** (`summarize_preferences`) जे वापरकर्त्याच्या महत्त्वाच्या तपशीलांचे संक्षिप्त रेकॉर्ड तयार करते
- एक **बहु-टर्न संभाषण** जे संदर्भ जपणे आणि बदल हाताळणे दाखवते

### प्रत्यक्ष जगातील उपयोग
- **ग्राहक सेवा बोट्स**: दीर्घकाळ चालणाऱ्या समर्थन सत्रांमध्ये प्राधान्ये लक्षात ठेवणे
- **वैयक्तिक सहाय्यक**: संदर्भ पुन्हा समजावून सांगितल्याशिवाय चालू प्रकल्प ट्रॅक करणे
- **शैक्षणिक शिक्षक**: अनेक संवादांमध्ये विद्यार्थ्याची प्रगती जपणे

### पुढील पावले
- फाइल-आधारित टिकाऊपणासह पूर्ण स्क्रॅचपॅड साधन अमलात आणा
- सारांश तयार केल्यावर स्वयंचलित इतिहास कटिंग जोडा
- अर्थपूर्ण स्मरणशक्तीसाठी व्हेक्टर डेटाबेससह एकत्र करा
- पूर्ण संदर्भासह अनेक दिवसांनी संभाषण सुरू करू शकणारे एजंट तयार करा


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**सूचना**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) च्या माध्यमातून भाषांतरित केला आहे. आम्ही अचूकतेसाठी प्रयत्नशील असलो तरी, ऑटोमेटेड भाषांतरांमध्ये चुका किंवा अचूकतेत त्रुटी असू शकतात याची कृपया जाणीव ठेवा. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्त्रोत मानला जावा. महत्त्वपूर्ण माहितीकरिता व्यावसायिक मानवी भाषांतर करण्याची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थान्वयासाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
